# 0. Load in the cleaned test and train data

In [10]:
import pandas as pd

test = pd.read_csv("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_test_cleaned.csv")
test.info()
submission = test[["GuestID"]].copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 60 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   GuestID                     1739 non-null   int64
 1   AllInclusive                1739 non-null   int64
 2   VIP                         1739 non-null   int64
 3   RoomService                 1739 non-null   int64
 4   Dining                      1739 non-null   int64
 5   Retail                      1739 non-null   int64
 6   Spa                         1739 non-null   int64
 7   Entertainment               1739 non-null   int64
 8   LoyaltyPoints               1739 non-null   int64
 9   SurveyScore                 1739 non-null   int64
 10  DaysSinceEmail              1739 non-null   int64
 11  SharedRoom                  1739 non-null   int64
 12  RoomNumber                  1739 non-null   int64
 13  BookingMonth                1739 non-null   int64
 14  BookingY

In [11]:
test.isna().sum()

GuestID                       0
AllInclusive                  0
VIP                           0
RoomService                   0
Dining                        0
Retail                        0
Spa                           0
Entertainment                 0
LoyaltyPoints                 0
SurveyScore                   0
DaysSinceEmail                0
SharedRoom                    0
RoomNumber                    0
BookingMonth                  0
BookingYear                   0
PromoCode_PromoA              0
PromoCode_PromoB              0
Region_AsiaPacific            0
Region_Europe                 0
Region_Unknown                0
PackageType_NoPackage         0
PackageType_Relaxation        0
PackageType_Wellness          0
AgeGroup_Middle               0
AgeGroup_Minor                0
AgeGroup_Senior               0
AgeGroup_Unknown              0
AgeGroup_Young                0
RoomFloor_B                   0
RoomFloor_C                   0
RoomFloor_D                   0
RoomFloo

In [12]:
train = pd.read_csv("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_train_cleaned.csv")
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 61 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   GuestID                     6954 non-null   int64
 1   AllInclusive                6954 non-null   int64
 2   VIP                         6954 non-null   int64
 3   RoomService                 6954 non-null   int64
 4   Dining                      6954 non-null   int64
 5   Retail                      6954 non-null   int64
 6   Spa                         6954 non-null   int64
 7   Entertainment               6954 non-null   int64
 8   LoyaltyPoints               6954 non-null   int64
 9   SurveyScore                 6954 non-null   int64
 10  DaysSinceEmail              6954 non-null   int64
 11  Churned                     6954 non-null   int64
 12  SharedRoom                  6954 non-null   int64
 13  RoomNumber                  6954 non-null   int64
 14  BookingM

In [13]:
train.isna().sum()

GuestID                       0
AllInclusive                  0
VIP                           0
RoomService                   0
Dining                        0
                             ..
ReferralSource_TikTok         0
ReferralSource_TripAdvisor    0
ReferralSource_Twitter        0
ReferralSource_Yelp           0
ReferralSource_YouTube        0
Length: 61, dtype: int64

In [14]:
#Create X and y variables for modeling

from sklearn.model_selection import train_test_split

X = train.drop('Churned', axis=1)
y = train['Churned']


# 1. Train the model - This is where your model goes

In [15]:
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

# CatBoost handles categoricals natively — just tell it which columns are categorical.
# GuestID is an identifier, drop it from features.
drop_cols = ["GuestID", "Churned"]
X = train.drop(columns=drop_cols)
y = train["Churned"]

cat_features = [c for c in X.columns if X[c].dtype == "object"]
# CatBoost requires categorical columns to have no NaN — fill with a sentinel string.
X[cat_features] = X[cat_features].fillna("missing")

from sklearn.model_selection import train_test_split
X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

neg, pos = (y_train_cb == 0).sum(), (y_train_cb == 1).sum()

cat = CatBoostClassifier(
    iterations=1000,
    depth=4,
    learning_rate=0.11169462714062046,
    l2_leaf_reg=0.0312574733854805,
    bagging_temperature=0.9697202489475304,
    random_strength=0.228241181807281,
    border_count=180,
    min_data_in_leaf=70,
    scale_pos_weight=neg / pos,
    eval_metric="AUC",
    early_stopping_rounds=50,
    random_seed=42,
    verbose=False,
)

train_pool = Pool(X_train_cb, y_train_cb, cat_features=cat_features)
test_pool = Pool(X_test_cb, y_test_cb, cat_features=cat_features)

cat.fit(train_pool, eval_set=test_pool)

probs = cat.predict(X_test_cb)
print(f"Test ROC-AUC:        {roc_auc_score(y_test_cb, probs):.3f}")
print(f"Best iteration:      {cat.get_best_iteration()}")

#use scitkit-learn classification report to get precision, recall, f1-score, and support for the test set predictions
from sklearn.metrics import classification_report
y_pred = cat.predict(X_test_cb)
print(classification_report(y_test_cb, y_pred))

# Feature importance
import pandas as pd
fi = pd.DataFrame({
    "feature": X_train_cb.columns,
    "importance": cat.get_feature_importance(train_pool),
}).sort_values("importance", ascending=False)
print(fi.to_string(index=False))

Test ROC-AUC:        0.807
Best iteration:      115
              precision    recall  f1-score   support

           0       0.80      0.82      0.81       690
           1       0.82      0.79      0.80       701

    accuracy                           0.81      1391
   macro avg       0.81      0.81      0.81      1391
weighted avg       0.81      0.81      0.81      1391

                   feature  importance
              AllInclusive   19.977782
                       Spa   10.522737
             Region_Europe    7.720946
             Entertainment    7.077440
               RoomFloor_F    4.420405
                RoomNumber    3.733373
               RoomService    3.728166
          PromoCode_PromoB    3.716023
          PromoCode_PromoA    3.324690
     ReferralSource_Friend    3.281761
     ReferralSource_TikTok    3.171939
            AgeGroup_Minor    3.032176
                    Dining    2.608650
        Region_AsiaPacific    2.130171
                    Retail    2.0314

# 2. Test - This will produce the test data

In [16]:
test.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 60 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   GuestID                     1739 non-null   int64
 1   AllInclusive                1739 non-null   int64
 2   VIP                         1739 non-null   int64
 3   RoomService                 1739 non-null   int64
 4   Dining                      1739 non-null   int64
 5   Retail                      1739 non-null   int64
 6   Spa                         1739 non-null   int64
 7   Entertainment               1739 non-null   int64
 8   LoyaltyPoints               1739 non-null   int64
 9   SurveyScore                 1739 non-null   int64
 10  DaysSinceEmail              1739 non-null   int64
 11  SharedRoom                  1739 non-null   int64
 12  RoomNumber                  1739 non-null   int64
 13  BookingMonth                1739 non-null   int64
 14  BookingY

In [17]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 59 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   AllInclusive                6954 non-null   int64
 1   VIP                         6954 non-null   int64
 2   RoomService                 6954 non-null   int64
 3   Dining                      6954 non-null   int64
 4   Retail                      6954 non-null   int64
 5   Spa                         6954 non-null   int64
 6   Entertainment               6954 non-null   int64
 7   LoyaltyPoints               6954 non-null   int64
 8   SurveyScore                 6954 non-null   int64
 9   DaysSinceEmail              6954 non-null   int64
 10  SharedRoom                  6954 non-null   int64
 11  RoomNumber                  6954 non-null   int64
 12  BookingMonth                6954 non-null   int64
 13  BookingYear                 6954 non-null   int64
 14  PromoCod

In [18]:
#run the test data against the model to get the probabilities for the submission file

#drop GueestID Column from the test data
test = test.drop(columns=["GuestID"])
probs = cat.predict(test)

#probs = xgb.predict(test)

#display the probs
print(probs)

#remove all columns except for the GuestID column and then add the probabilities as a new column called "Churned"
#submission = train[["GuestID"]].copy()
submission["Churned"] = probs

submission.info()

#output the csv file with the predictions
submission.to_csv("submission.csv", index=False)

[1 0 0 ... 1 0 0]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   GuestID  1739 non-null   int64
 1   Churned  1739 non-null   int64
dtypes: int64(2)
memory usage: 27.3 KB
